# Ensemble proprioception pipeline

This notebook runs path generation, IK, motion, and muscle extraction once. It then sweeps independently over spindle-coefficient realizations and pretrained CNN checkpoints, finally saving the mean and standard deviation of every predicted channel.

The sweep is a Cartesian product: every spindle seed is evaluated with every checkpoint listed in `MODEL_CHECKPOINTS`. Repeating an identical pair is unnecessary because inference is deterministic.

In [ ]:
from pathlib import Path
import copy
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from run_pipeline import Pipeline

ROOT = Path.cwd().resolve()
BASE_CONFIG = ROOT / 'experiments' / 'center_out.yaml'
EXPERIMENT = 'center_out_ensemble'
RUN_DETERMINISTIC_STAGES = True
DISPLAY_SIMULATION = False

# Input spindle populations to sample. The repository ships seeds 0-4.
SPINDLE_COEFFICIENT_SEEDS = [0, 1, 2, 3, 4]

# CNNs to evaluate, as (coefficient seed used to train CNN, training seed).
# Keeping this list at [(0, 9)] fixes the model while spindle coefficients vary.
MODEL_CHECKPOINTS = [(0, 9)]

TEMPORAL_PADDING_MODE = 'replicate'
MODEL_PATH = None  # None derives each checkpoint path from its two seeds.

assert TEMPORAL_PADDING_MODE in {'zeros', 'replicate', 'reflect'}
assert SPINDLE_COEFFICIENT_SEEDS and MODEL_CHECKPOINTS

## Prepare a sweep-specific configuration

The generated YAML lives inside the experiment output directory. Change the base configuration or parameters above, then rerun this cell.

In [ ]:
with BASE_CONFIG.open(encoding='utf-8') as handle:
    base_config = yaml.safe_load(handle)

base_config['experiment'] = EXPERIMENT
base_config['display_simulation'] = DISPLAY_SIMULATION
base_config.setdefault('proprioception', {})['temporal_padding_mode'] = TEMPORAL_PADDING_MODE

experiment_dir = ROOT / 'outputs' / EXPERIMENT
experiment_dir.mkdir(parents=True, exist_ok=True)
sweep_config_path = experiment_dir / 'ensemble_config.yaml'

def write_run_config(spindle_seed, model_coefficient_seed, training_seed):
    config = copy.deepcopy(base_config)
    proprio = config['proprioception']
    proprio['coefficient_seed'] = int(spindle_seed)
    proprio['model_coefficient_seed'] = int(model_coefficient_seed)
    proprio['training_seed'] = int(training_seed)
    proprio['model_path'] = MODEL_PATH
    with sweep_config_path.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(config, handle, sort_keys=False)
    return Pipeline(sweep_config_path)

# Initialize the file with the first requested realization.
write_run_config(SPINDLE_COEFFICIENT_SEEDS[0], *MODEL_CHECKPOINTS[0])
print(f'Configuration: {sweep_config_path}')
print(f'Outputs:       {experiment_dir}')

## Deterministic stages (once)

These stages produce the paths, joint trajectories, motions, and muscle lengths. Set `RUN_DETERMINISTIC_STAGES = False` when resuming a completed sweep without changing any of those inputs.

In [ ]:
pipeline = write_run_config(SPINDLE_COEFFICIENT_SEEDS[0], *MODEL_CHECKPOINTS[0])
if RUN_DETERMINISTIC_STAGES:
    pipeline.run_through(through='muscle-lengths', start='path', display=DISPLAY_SIMULATION)
else:
    required = [experiment_dir / 'manifest.yaml', experiment_dir / 'muscles']
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f'Missing cached deterministic outputs: {missing}')
    print('Reusing cached deterministic outputs.')

## Stochastic-parameter sweep

For each spindle seed, muscle signals are generated once. CNN inference is then run for every requested checkpoint. Each run is copied into memory before the pipeline files are overwritten by the next realization.

In [ ]:
runs = {}
prediction_dir = experiment_dir / 'predictions'

for spindle_seed in SPINDLE_COEFFICIENT_SEEDS:
    first_model_seed, first_train_seed = MODEL_CHECKPOINTS[0]
    pipeline = write_run_config(spindle_seed, first_model_seed, first_train_seed)
    pipeline.run_stage('muscle-signals')

    for model_coefficient_seed, training_seed in MODEL_CHECKPOINTS:
        pipeline = write_run_config(spindle_seed, model_coefficient_seed, training_seed)
        pipeline.run_stage('inference')
        run_key = (int(spindle_seed), int(model_coefficient_seed), int(training_seed))
        per_direction = {}
        for csv_path in sorted(prediction_dir.glob('*.csv')):
            if csv_path.name != 'summary.csv':
                per_direction[csv_path.stem] = pd.read_csv(csv_path)
        if not per_direction:
            raise RuntimeError(f'Inference produced no per-direction CSVs for {run_key}')
        runs[run_key] = per_direction
        print(f'Captured spindle/model/train seeds: {run_key}')

print(f'Completed {len(runs)} realizations.')

## Average predictions and save uncertainty

Ground-truth and time columns are copied once. Predicted channels receive both `_mean` and `_std` columns. Wrist L2 is recomputed from the ensemble-mean XYZ prediction rather than averaging each run's L2.

In [ ]:
ensemble_dir = experiment_dir / 'ensemble_predictions'
ensemble_dir.mkdir(parents=True, exist_ok=True)
run_keys = list(runs)
directions = sorted(runs[run_keys[0]])
prediction_columns = [
    'pred_wrist_X_cm', 'pred_wrist_Y_cm', 'pred_wrist_Z_cm',
    'pred_shoulder_elv_deg', 'pred_shoulder_rot_deg', 'pred_elbow_flexion_deg',
]
truth_columns = [column for column in runs[run_keys[0]][directions[0]].columns
                 if column == 'time_s' or column.startswith('true_')]
ensemble = {}

for direction in directions:
    frames = [runs[key][direction] for key in run_keys]
    lengths = {len(frame) for frame in frames}
    if len(lengths) != 1:
        raise ValueError(f'{direction}: realization lengths differ: {lengths}')
    result = frames[0][truth_columns].copy()
    for column in prediction_columns:
        values = np.stack([frame[column].to_numpy() for frame in frames])
        result[f'{column}_mean'] = values.mean(axis=0)
        result[f'{column}_std'] = values.std(axis=0, ddof=0)
    xyz_error = np.column_stack([
        result['pred_wrist_X_cm_mean'] - result['true_wrist_X_cm'],
        result['pred_wrist_Y_cm_mean'] - result['true_wrist_Y_cm'],
        result['pred_wrist_Z_cm_mean'] - result['true_wrist_Z_cm'],
    ])
    result['ensemble_mean_l2_distance_cm'] = np.linalg.norm(xyz_error, axis=1)
    result.to_csv(ensemble_dir / f'{direction}.csv', index=False)
    ensemble[direction] = result

metadata = {
    'spindle_coefficient_seeds': SPINDLE_COEFFICIENT_SEEDS,
    'model_checkpoints': [list(pair) for pair in MODEL_CHECKPOINTS],
    'temporal_padding_mode': TEMPORAL_PADDING_MODE,
    'number_of_realizations': len(run_keys),
}
with (ensemble_dir / 'ensemble_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Saved ensemble predictions to {ensemble_dir}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for direction, frame in ensemble.items():
    true_x = frame['true_wrist_X_cm'] - frame['true_wrist_X_cm'].iloc[0]
    true_y = frame['true_wrist_Y_cm'] - frame['true_wrist_Y_cm'].iloc[0]
    pred_x = frame['pred_wrist_X_cm_mean'] - frame['pred_wrist_X_cm_mean'].iloc[0]
    pred_y = frame['pred_wrist_Y_cm_mean'] - frame['pred_wrist_Y_cm_mean'].iloc[0]
    axes[0].plot(true_x, true_y, label=direction)
    axes[1].plot(pred_x, pred_y, label=direction)
for axis, title in zip(axes, ['Truth', f'Ensemble mean (n={len(run_keys)})']):
    axis.set_title(title)
    axis.set_xlabel('X displacement (cm)')
    axis.set_aspect('equal')
    axis.grid(alpha=0.2)
axes[0].set_ylabel('Y displacement (cm)')
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()
figure_path = ensemble_dir / 'ensemble_trajectory_comparison.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()
print(figure_path)